In [2]:
import tensorflow as tf
from tensorflow.keras.models import load_model 
import numpy as np
import pickle
import pandas as pd

In [ ]:
#load the model
model = load_model('churn_model.h5',compile=False)

#load the enocder and scaler
with open('label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)
with open('onehot_encoder.pkl', 'rb') as f:
    onehot_encoder = pickle.load(f)
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

In [5]:
#Example input data for prediction
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [6]:
#one-hot encode the categorical features
geography_encoded = onehot_encoder.transform([[input_data['Geography']]]).toarray()
geography_df = pd.DataFrame(geography_encoded, columns=onehot_encoder.get_feature_names_out(['Geography']))
geography_df 

c:\Users\sivar\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [7]:
# Convert input data to DataFrame
input_df = pd.DataFrame([input_data])
input_df 

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [8]:
input_df = pd.concat([input_df.drop('Geography', axis=1), geography_df], axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,Male,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [9]:
input_df['Gender'] = label_encoder.transform(input_df['Gender'])
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [10]:
input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [11]:
model_prediction = model.predict(input_scaled)
model_prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 269ms/step


array([[0.04482955]], dtype=float32)

In [14]:
prediction_probability = model_prediction[0][0]
prediction_class = 1 if prediction_probability > 0.5 else 0
print(f"Prediction Probability: {prediction_probability}")
print(f"Prediction Class: {prediction_class}")

Prediction Probability: 0.04482954740524292
Prediction Class: 0
